# PTEN Hands-on Lab A: from drilling ML to a DSA agent in a container

**Goal (60 min):** take the time-series drilling data you already produce, build a stuck-pipe
risk model on it, put a Cortex Agent on top, and serve that agent from a containerized app,
all inside Snowflake.

| Block | What you do | Snowsight surface you visit |
|---|---|---|
| A0 | Frame the problem, look at the data | Catalog / Database Explorer |
| A1 | Stand-level features into the Feature Store | AI & ML » Features |
| A2 | Train a model, track the experiment | AI & ML » Experiments |
| A3 | Register the model, batch score | AI & ML » Models |
| A3b | Insights: MSE, flat time, drag, rig state; serve the model on SPCS | AI & ML » Models |
| A4 | Semantic view, tools, DSA agent | AI & ML » Agents |
| A5 | Serve the agent from SPCS, with feedback | Monitoring » Services |
| A6 | Wrap up, follow-ups, bridge to Lab B | |

**How to use this notebook:** run it top to bottom (**Run all** works). The first code cells set up
the account, so nothing needs to be run beforehand. Sections marked **Exercise for later** are left
open on purpose: they are the next steps for your own data.
Each section also has a **Vendor agnostic vs Snowflake native** callout, because the right
answer is not always the same and you should be able to defend the choice.

**Two ways through every step:** each major step opens with **why it matters** on the rig and
**two ways to run it**: (A) run the Python/SQL cells as written, or (B) hand the step to **Cortex Code**
with the sample prompt shown. Option A teaches the APIs; option B shows how a data scientist gets the
same result in a sentence.

> The data is **synthetic**, shaped like PTEN's: a tall/skinny table of 1-minute values
> (rig, well, timestamp, tag, value, state, stand), the kind of output QuestDB lands as Iceberg
> today. 20 rigs, 40 tags, 14 days, about 16M rows. Stuck-pipe precursors are injected before
> labeled events so the model has real signal to find.

## A0. Setup and a first look at the data  *(Run now)*

> **Why it matters:** PTEN already lands QuestDB 1-minute aggregates as a tall/skinny table. We start by showing that the raw shape is fine to work with in place: no export, no reshaping job.
>
> **Two ways to run this step**
> - **Option A, plain Python/SQL:** run the cells below as written.
> - **Option B, Cortex Code (CoCo):** open CoCo (Snowsight Workspaces, Desktop or CLI), the `sql-author` skill
>   loads automatically, then paste:
>
> ```text
> Profile HOL_PTEN_ML.RAW.TAG_VALUE: row counts by tag and state, and plot torque, hookload, ROP and SPP for the 3 hours before the first STUCK_PIPE event in HOL_PTEN_ML.LABELS.NPT_EVENTS.
> ```
>
> Both paths create the same object names, so you can mix them and the later cells still work.

In [ ]:
# Session + helpers. Works in a Snowsight Workspace notebook (active session) or locally
# with a named connection (SNOWFLAKE_CONNECTION_NAME).
import os, re
from pathlib import Path
try:
    from snowflake.snowpark.context import get_active_session
    session = get_active_session()
except Exception:
    from snowflake.snowpark import Session
    session = Session.builder.config("connection_name", os.getenv("SNOWFLAKE_CONNECTION_NAME", "default")).create()


def q(sql):
    """Run SQL and return a pandas DataFrame."""
    if sql.lstrip().upper().startswith(("SHOW", "DESC")):
        # to_pandas() on some SHOW outputs (e.g. SHOW SERVICES) fails on a column-count
        # mismatch; reading the result back through RESULT_SCAN is reliable.
        session.sql(sql).collect()
        return session.sql("SELECT * FROM TABLE(RESULT_SCAN(LAST_QUERY_ID()))").to_pandas()
    return session.sql(sql).to_pandas()

REPO = Path.cwd().parent if Path.cwd().name.startswith("lab-") else Path.cwd()

def run_sql_file(rel):
    """Run a repo SQL file. Splits on ';' at line ends, keeping $$ ... $$ bodies intact."""
    text = (REPO / rel).read_text()
    stmts, buf, in_body = [], [], False
    for line in text.splitlines():
        if line.strip().startswith("--") and not in_body:
            continue
        buf.append(line)
        if line.count("$$") % 2 == 1:
            in_body = not in_body
        # a trailing "-- comment" after the ';' still ends the statement
        if not in_body and re.sub(r"\s--.*$", "", line).rstrip().endswith(";"):
            buf[-1] = re.sub(r"\s--.*$", "", buf[-1])
            stmts.append("\n".join(buf).strip().rstrip(";")); buf = []
    last = None
    for s in stmts:
        if s:
            last = session.sql(s).collect()
    print(f"ran {len(stmts)} statements from {rel}")
    return last

def use_lab():
    session.sql("USE WAREHOUSE HOL_PTEN_WH").collect()
    session.sql("USE DATABASE HOL_PTEN_ML").collect()
    print(session.get_current_account(), session.get_current_role(), "-> HOL_PTEN_WH / HOL_PTEN_ML")

In [ ]:
# One-time account setup, about 2 minutes. Safe to re-run: existing objects are kept and the
# synthetic data is only generated once.
#   setup/00_setup.sql             role, warehouse, compute pool, PyPI access, database and schemas
#   setup/01_generate_synthetic.sql  about 16M rows of 1-minute tag data, aggregates, NPT events
session.sql("USE ROLE ACCOUNTADMIN").collect()
run_sql_file("setup/00_setup.sql")
have_data = session.sql("""SELECT COUNT(*) FROM HOL_PTEN_ML.INFORMATION_SCHEMA.TABLES
                           WHERE TABLE_SCHEMA = 'RAW' AND TABLE_NAME = 'TAG_VALUE'""").collect()[0][0]
if have_data:
    print("synthetic data already present, skipping 01_generate_synthetic.sql")
else:
    run_sql_file("setup/01_generate_synthetic.sql")
use_lab()

In [ ]:
q("""
SELECT 'RAW.TAG_VALUE' AS table_name, COUNT(*) AS row_count FROM RAW.TAG_VALUE
UNION ALL SELECT 'AGG.STAND_STATE_TAG_AGG', COUNT(*) FROM AGG.STAND_STATE_TAG_AGG
UNION ALL SELECT 'LABELS.NPT_EVENTS', COUNT(*) FROM LABELS.NPT_EVENTS
""")

In [ ]:
# One rig, two hours, four channels: what a stuck-pipe precursor looks like in the raw data
ev = q("SELECT rig_id, event_start_ts FROM LABELS.NPT_EVENTS WHERE event_type='STUCK_PIPE' ORDER BY event_start_ts LIMIT 1")
rig, t0 = ev.RIG_ID[0], ev.EVENT_START_TS[0]
win = q(f"""
SELECT ts, tag, value FROM RAW.TAG_VALUE
WHERE rig_id = '{rig}' AND tag IN ('TORQUE_KFTLB','HOOKLOAD_KLB','ROP_FPH','SPP_PSI')
  AND ts BETWEEN DATEADD(MINUTE, -180, '{t0}') AND '{t0}' AND state = 'DRILLING'
""")
print(f"{rig}: stuck pipe at {t0}")
win.pivot_table(index="TS", columns="TAG", values="VALUE").plot(subplots=True, figsize=(10, 7))

**What to notice:** in the last two hours before the event, torque and hookload drift up while
ROP drops. That is the classic pack-off signature the field already knows. The model's job is to
spot it consistently across 100 rigs, not just on the one a driller happens to be watching.

### The DSA question this lab answers
*"Which rigs are trending toward stuck pipe, what do the sensors show, and what should the
crew check?"* Lab A covers the first two parts. Lab B adds the third (documents).

> **Vendor agnostic vs Snowflake native: where the time-series work happens**
>
> | | Vendor agnostic | Snowflake native |
> |---|---|---|
> | **How** | Extract to Parquet/S3, process with pandas or Spark on EC2/EMR/Databricks | Query the tall/skinny table in place with SQL and Snowpark; Iceberg tables work the same way |
> | **Pros** | Any engine, any library; teams already know pandas | No copies, one governance model, elastic compute per query, same data the BI and agent use |
> | **Cons** | Copies of the data, separate security model, cluster ops, egress | Snowflake compute pricing; some niche libraries need container runtime |
>
> **When to pick which:** keep heavy, exotic signal processing wherever it already runs well, but do aggregation, feature engineering and anything the agent must query next to the data.

## A1. Features per stand, into the Feature Store  *(Run now, about 10 min)*

**Drilling context:** engineers think in **stands** (about 90 ft of pipe, about 90 minutes of
drilling) and **rig states** (drilling, connection, circulating, tripping). A torque average over
a whole day mixes on-bottom and off-bottom values and hides the precursor. So the features are
per stand, for the DRILLING state only: mean, max, variability and trend (slope) of hookload,
torque, ROP, RPM, SPP and WOB.

The aggregates already exist in `AGG.STAND_STATE_TAG_AGG` (one row per rig, stand, state and tag),
which mirrors the QuestDB 1-minute aggregates. We pivot the tags we care about into columns.

> **Why it matters:** Features are where drilling knowledge becomes ML. We compute them once, per stand and rig state, and register them so the next model (and the next data scientist) reuses them instead of re-deriving them.
>
> **Two ways to run this step**
> - **Option A, plain Python/SQL:** run the cells below as written.
> - **Option B, Cortex Code (CoCo):** open CoCo (Snowsight Workspaces, Desktop or CLI), the `machine-learning` skill
>   loads automatically, then paste:
>
> ```text
> Create a Feature Store FEATURE_STORE in HOL_PTEN_ML with entity RIG_STAND (join keys RIG_ID, STAND_NO) and feature view STAND_DRILLING_FEATURES V1 over HOL_PTEN_ML.AGG.STAND_STATE_TAG_AGG for state DRILLING: mean, max, std and slope of hookload, torque, ROP, RPM, SPP and WOB per stand, timestamp STAND_END_TS, daily refresh.
> ```
>
> Both paths create the same object names, so you can mix them and the later cells still work.

In [ ]:
FEATURE_SQL = """
SELECT rig_id, stand_no, MAX(end_ts) AS stand_end_ts,
  AVG(IFF(tag='HOOKLOAD_KLB', avg_value, NULL))     AS hookload_avg,
  AVG(IFF(tag='HOOKLOAD_KLB', max_value, NULL))     AS hookload_max,
  AVG(IFF(tag='HOOKLOAD_KLB', slope_per_min, NULL)) AS hookload_slope,
  AVG(IFF(tag='TORQUE_KFTLB', avg_value, NULL))     AS torque_avg,
  AVG(IFF(tag='TORQUE_KFTLB', std_value, NULL))     AS torque_std,
  AVG(IFF(tag='TORQUE_KFTLB', slope_per_min, NULL)) AS torque_slope,
  AVG(IFF(tag='ROP_FPH', avg_value, NULL))          AS rop_avg,
  AVG(IFF(tag='ROP_FPH', slope_per_min, NULL))      AS rop_slope,
  AVG(IFF(tag='RPM', avg_value, NULL))              AS rpm_avg,
  AVG(IFF(tag='SPP_PSI', avg_value, NULL))          AS spp_avg,
  AVG(IFF(tag='WOB_KLB', avg_value, NULL))          AS wob_avg
FROM HOL_PTEN_ML.AGG.STAND_STATE_TAG_AGG
WHERE state = 'DRILLING'
GROUP BY rig_id, stand_no
"""
q(FEATURE_SQL + " ORDER BY rig_id, stand_no LIMIT 5")

In [ ]:
from snowflake.ml.feature_store import FeatureStore, Entity, FeatureView, CreationMode

fs = FeatureStore(session, database="HOL_PTEN_ML", name="FEATURE_STORE",
                  default_warehouse="HOL_PTEN_WH", creation_mode=CreationMode.CREATE_IF_NOT_EXIST)
stand = Entity(name="RIG_STAND", join_keys=["RIG_ID", "STAND_NO"], desc="One drill-pipe stand on one rig")
fs.register_entity(stand)

fv = FeatureView(name="STAND_DRILLING_FEATURES", entities=[stand],
                 feature_df=session.sql(FEATURE_SQL), timestamp_col="STAND_END_TS",
                 refresh_freq="1 day",          # backed by a dynamic table that refreshes itself
                 desc="Per-stand drilling channel statistics (DRILLING state)")
fv = fs.register_feature_view(fv, version="V1", overwrite=True)
print(fv.fully_qualified_name())

**Go look:** Snowsight » **AI & ML » Features**. You will see the `RIG_STAND` entity and the
`STAND_DRILLING_FEATURES` view, its lineage back to the aggregate table, and its refresh schedule.
A second data scientist can now reuse these features instead of re-deriving them.

> **Vendor agnostic vs Snowflake native: feature engineering and reuse**
>
> | | Vendor agnostic | Snowflake native |
> |---|---|---|
> | **How** | Feast or a homegrown feature table; pandas rolling windows in each notebook | Snowflake Feature Store: entities + feature views backed by dynamic tables |
> | **Pros** | Portable across clouds; Feast is open source | Incremental refresh is managed; point-in-time correct training sets; lineage and RBAC built in |
> | **Cons** | Separate offline/online stores to run; point-in-time joins are easy to get wrong; features drift between notebooks | Snowflake-specific API; online serving needs the online store option |
>
> **When to pick which:** if features must be served to systems outside Snowflake at millisecond latency, check the online store first; for batch scoring and agent use, native is less to operate.

> **Exercise for later: rolling windows across stands**
>
> Add a second feature view with 3-stand rolling deltas (torque trend vs the previous 3 stands). Hint: window functions over `STAND_NO` partitioned by `RIG_ID`, then join both views in `generate_training_set`. Does PR-AUC improve?

## A2. Train a stuck-pipe model and track the experiment  *(Run now, about 12 min)*

**Label:** a stand is positive when a STUCK_PIPE event starts within **3 hours** after the stand
ends. That is a lead time a crew can act on (circulate, ream, adjust parameters).

**Split:** time-based (first 70 percent of the window trains, the rest tests). Random splits leak
the future into training on time series and make the model look better than it is.

> **Why it matters:** The label is the business question: will this rig get stuck in the next 3 hours? Everything else is plumbing. We track every run so the choice of model is auditable, not a notebook nobody can reproduce.
>
> **Two ways to run this step**
> - **Option A, plain Python/SQL:** run the cells below as written.
> - **Option B, Cortex Code (CoCo):** open CoCo (Snowsight Workspaces, Desktop or CLI), the `machine-learning` skill
>   loads automatically, then paste:
>
> ```text
> Label each stand in HOL_PTEN_ML.ML.STAND_LABELS as 1 when a STUCK_PIPE event starts within 180 minutes after it ends, build a training set from feature view STAND_DRILLING_FEATURES V1, train an XGBoost classifier with a time-based 70/30 split, and log runs xgb_baseline and xgb_deeper with ROC-AUC and PR-AUC to experiment STUCK_PIPE_EXPERIMENT.
> ```
>
> Both paths create the same object names, so you can mix them and the later cells still work.

In [ ]:
session.sql("""
CREATE OR REPLACE TABLE HOL_PTEN_ML.ML.STAND_LABELS AS
SELECT s.rig_id, s.stand_no, s.stand_end_ts,
       MAX(IFF(e.event_id IS NOT NULL, 1, 0)) AS stuck_pipe_next_3h
FROM (SELECT rig_id, stand_no, MAX(end_ts) AS stand_end_ts
      FROM HOL_PTEN_ML.AGG.STAND_STATE_TAG_AGG WHERE state = 'DRILLING' GROUP BY 1, 2) s
LEFT JOIN HOL_PTEN_ML.LABELS.NPT_EVENTS e
  ON e.rig_id = s.rig_id AND e.event_type = 'STUCK_PIPE'
 AND DATEDIFF(MINUTE, s.stand_end_ts, e.event_start_ts) BETWEEN 0 AND 180
GROUP BY 1, 2, 3
""").collect()

train_df = fs.generate_training_set(
    spine_df=session.table("HOL_PTEN_ML.ML.STAND_LABELS"), features=[fv],
    spine_timestamp_col="STAND_END_TS", spine_label_cols=["STUCK_PIPE_NEXT_3H"]).to_pandas()

FEATS = [c for c in train_df.columns if c not in ("RIG_ID", "STAND_NO", "STAND_END_TS", "STUCK_PIPE_NEXT_3H")]
cut = train_df["STAND_END_TS"].quantile(0.7)
tr, te = train_df[train_df.STAND_END_TS <= cut], train_df[train_df.STAND_END_TS > cut]
print(f"train {len(tr)}  test {len(te)}  positives {int(train_df.STUCK_PIPE_NEXT_3H.sum())}")

In [ ]:
from xgboost import XGBClassifier
from sklearn.metrics import roc_auc_score, average_precision_score
from snowflake.ml.experiment import ExperimentTracking

exp = ExperimentTracking(session, database_name="HOL_PTEN_ML", schema_name="ML")
exp.set_experiment("STUCK_PIPE_EXPERIMENT")

def train_and_log(run_name, **params):
    params = {"n_estimators": 200, "max_depth": 4, "learning_rate": 0.05,
              "scale_pos_weight": float((tr.STUCK_PIPE_NEXT_3H == 0).sum() / tr.STUCK_PIPE_NEXT_3H.sum()), **params}
    model = XGBClassifier(**params).fit(tr[FEATS], tr.STUCK_PIPE_NEXT_3H)
    p = model.predict_proba(te[FEATS])[:, 1]
    metrics = {"roc_auc": roc_auc_score(te.STUCK_PIPE_NEXT_3H, p),
               "pr_auc": average_precision_score(te.STUCK_PIPE_NEXT_3H, p)}
    with exp.start_run(run_name):
        exp.log_params(params); exp.log_metrics(metrics)
    print(run_name, {k: round(v, 3) for k, v in metrics.items()})
    return model, metrics

model, metrics = train_and_log("xgb_baseline")
_ = train_and_log("xgb_deeper", max_depth=6, n_estimators=400)

**Go look:** Snowsight » **AI & ML » Experiments » STUCK_PIPE_EXPERIMENT**. Select both runs and
**Compare**. On imbalanced events like stuck pipe, read **PR-AUC**, not accuracy: a model that
always says "no stuck pipe" is 97 percent accurate and useless.

> **Vendor agnostic vs Snowflake native: training and experiment tracking**
>
> | | Vendor agnostic | Snowflake native |
> |---|---|---|
> | **How** | Train on a laptop or EC2; MLflow tracking server for runs | Train in a Snowflake notebook (warehouse or container runtime, CPU or GPU); Snowflake Experiments |
> | **Pros** | MLflow is the de facto standard; works with any compute | Data never leaves; runs, metrics and models linked by lineage; no tracking server |
> | **Cons** | Tracking server, artifact store and auth to run; data copied out of the warehouse | Experiments UI is newer than MLflow's; MLflow models can still be imported into the registry |
>
> **When to pick which:** if the team is standardized on MLflow, keep it: the registry imports MLflow models. If not, native tracking removes a server nobody wants to own.

> **Exercise for later: tune and explain**
>
> Use Snowflake's distributed HPO (`snowflake.ml.modeling.tune`) over max_depth and learning_rate, then log the best model with `options={'enable_explainability': True}` and run `mv.run(..., function_name='explain')` to see which channels drive each risk score.

## A3. Register the model and batch score every stand  *(Run now, about 5 min)*

The registry stores the model as a governed Snowflake object with versions, metrics and an inference
function you can call from SQL. In A3b the same version is also served from SPCS, which is what the
agent's `ScoreRigNow` tool calls in A4.

> **Why it matters:** Once a model is in the registry it is a Snowflake object: versioned, permissioned, and callable from SQL. That is what lets an agent, a dashboard and a scheduled job all use the same model.
>
> **Two ways to run this step**
> - **Option A, plain Python/SQL:** run the cells below as written.
> - **Option B, Cortex Code (CoCo):** open CoCo (Snowsight Workspaces, Desktop or CLI), the `machine-learning` skill
>   loads automatically, then paste:
>
> ```text
> Register the best model from experiment STUCK_PIPE_EXPERIMENT as HOL_PTEN_ML.ML.STUCK_PIPE_MODEL version V1 for the warehouse, then score every row of feature view STAND_DRILLING_FEATURES V1 and save RIG_ID, STAND_NO, STAND_END_TS and the class-1 probability as STUCK_PIPE_RISK in HOL_PTEN_ML.PREDICTIONS.STUCK_PIPE_SCORES.
> ```
>
> Both paths create the same object names, so you can mix them and the later cells still work.

In [ ]:
from snowflake.ml.registry import Registry

reg = Registry(session, database_name="HOL_PTEN_ML", schema_name="ML")
mv = reg.log_model(model, model_name="STUCK_PIPE_MODEL", version_name="V1",
                   sample_input_data=tr[FEATS].head(50), target_platforms=["WAREHOUSE"],
                   metrics=metrics, comment="XGBoost stuck-pipe risk, next 3 hours, per stand")

scored = mv.run(fs.read_feature_view(fv), function_name="predict_proba")
scored.select("RIG_ID", "STAND_NO", "STAND_END_TS", scored['"output_feature_1"'].alias("STUCK_PIPE_RISK")) \
      .write.save_as_table("HOL_PTEN_ML.PREDICTIONS.STUCK_PIPE_SCORES", mode="overwrite")
q("SELECT * FROM PREDICTIONS.STUCK_PIPE_SCORES ORDER BY stuck_pipe_risk DESC LIMIT 5")

**Go look:** Snowsight » **AI & ML » Models » STUCK_PIPE_MODEL**: version V1, its metrics, its
lineage back to the feature view, and the SQL to call it:
`HOL_PTEN_ML.ML.STUCK_PIPE_MODEL!PREDICT_PROBA(...)`.

> **Vendor agnostic vs Snowflake native: model serving**
>
> | | Vendor agnostic | Snowflake native |
> |---|---|---|
> | **How** | Pickle on S3 + a Flask/SageMaker endpoint; cron job for batch scoring | Model Registry: SQL/Python inference in a warehouse, or a service on SPCS for real time |
> | **Pros** | Total control of the serving stack | Scoring is a SQL call next to the data; versioning, RBAC and lineage built in |
> | **Cons** | Another endpoint, IAM role and autoscaler to run; scores land somewhere else and must be copied back | Custom serving logic needs a custom model class or SPCS |
>
> **When to pick which:** batch and agent-driven scoring: native. Sub-100 ms online scoring at the rig edge: still an SPCS service or an edge deployment, fed by the same registry.

> **Exercise for later: schedule it**
>
> Wrap the scoring cell in a stored procedure and run it from a task every 15 minutes, or score incrementally from a stream on the feature view. Then add ML Observability to track drift by rig.

## A3b. More insights from the same pipeline  *(Run now, about 8 min)*

Stuck pipe is one of five things a drilling data scientist is asked about. With the same tag table
we can cover three more, cheaply, and hand all of them to the agent and the app:

| Operational pillar | Insight | How |
|---|---|---|
| Drilling optimization | **Mechanical Specific Energy (MSE)** per stand vs ROP | SQL, Teale's equation |
| Flat time | **Connection time** per rig vs fleet p50 / p90, hours lost | SQL |
| Wellbore stability | **Drag** (pick-up minus slack-off hookload) trend per connection | SQL |
| Rig state | **Rig-state classifier** per minute (drilling, connection, circulating, tripping) | XGBoost, registered |

The rig-state model is the quiet foundation: in production, state labels drive every per-state feature
above, so you want a model, not a rule, producing them.

> **Why it matters:** This is the point where the DSA team stops being a one-model shop. The same data, the same governance, four more answers the agent can give, and none of them needed a new pipeline.
>
> **Two ways to run this step**
> - **Option A, plain Python/SQL:** run the cells below (the first one runs `setup/02_insights.sql`).
> - **Option B, Cortex Code (CoCo):** open CoCo (Snowsight Workspaces, Desktop or CLI), the `sql-author` skill
>   loads automatically, then paste:
>
> ```text
> In HOL_PTEN_ML.AGG create STAND_MSE (Teale MSE in ksi per stand from WOB, torque, RPM, ROP while DRILLING, bit diameter from WELL_HEADER), CONNECTION_TIMES (minutes in CONNECTION per stand), a CONNECTION_BENCHMARK view (rig median vs fleet p50/p90 and hours above the fleet median), and OVERPULL_TREND (pick-up minus slack-off hookload per connection with a 5-connection rolling average).
> ```
>
> Both paths create the same object names, so you can mix them and the later cells still work.

In [ ]:
# MSE, connection benchmark and drag tables: runs setup/02_insights.sql statement by statement (open the file to read the SQL)
run_sql_file("setup/02_insights.sql")

In [ ]:
q("""SELECT rig_id, rig_median_min, fleet_p50_min, fleet_p90_min, hours_above_fleet_median
     FROM AGG.CONNECTION_BENCHMARK ORDER BY hours_above_fleet_median DESC LIMIT 5""")

**What to notice:** three rigs spend about 7 minutes more per connection than the fleet median. Over
two weeks that is about 25 hours each, more than the stuck-pipe NPT on most rigs. Flat time is often
the cheapest hours to win back.

> **Why it matters:** Rig state is the label every other model depends on. We train a small classifier on minutes where the state is known and score a held-out day, so the same pipeline can label rigs where it is not.
>
> **Two ways to run this step**
> - **Option A, plain Python/SQL:** run the cells below as written.
> - **Option B, Cortex Code (CoCo):** open CoCo (Snowsight Workspaces, Desktop or CLI), the `machine-learning` skill
>   loads automatically, then paste:
>
> ```text
> Pivot HOL_PTEN_ML.RAW.TAG_VALUE to one row per rig and minute with hookload, torque, SPP, RPM, ROP, WOB, block height, flow in and block height delta; train an XGBoost multi-class rig-state model on 2026-10-02, 10-05 and 10-08, score 2026-10-12, log accuracy to STUCK_PIPE_EXPERIMENT, register it as ML.RIG_STATE_CLASSIFIER V1 and write RIG_ID, TS, GIVEN_STATE, PREDICTED_STATE to PREDICTIONS.RIG_STATE_PREDICTIONS.
> ```
>
> Both paths create the same object names, so you can mix them and the later cells still work.

In [ ]:
import time
import pandas as pd
from sklearn.metrics import accuracy_score, confusion_matrix

WIDE_SQL = """
SELECT rig_id, ts, ANY_VALUE(state) AS state,
  MAX(IFF(tag='HOOKLOAD_KLB', value, NULL)) AS hookload, MAX(IFF(tag='TORQUE_KFTLB', value, NULL)) AS torque,
  MAX(IFF(tag='SPP_PSI', value, NULL)) AS spp, MAX(IFF(tag='RPM', value, NULL)) AS rpm,
  MAX(IFF(tag='ROP_FPH', value, NULL)) AS rop, MAX(IFF(tag='WOB_KLB', value, NULL)) AS wob,
  MAX(IFF(tag='BLOCK_HEIGHT_FT', value, NULL)) AS block_height, MAX(IFF(tag='FLOW_IN_GPM', value, NULL)) AS flow_in
FROM HOL_PTEN_ML.RAW.TAG_VALUE
WHERE tag IN ('HOOKLOAD_KLB','TORQUE_KFTLB','SPP_PSI','RPM','ROP_FPH','WOB_KLB','BLOCK_HEIGHT_FT','FLOW_IN_GPM')
  AND state <> 'NPT' AND TO_DATE(ts) IN ({days})
GROUP BY 1, 2"""

def wide(days):
    df = q(WIDE_SQL.format(days=days)).sort_values(["RIG_ID", "TS"])
    df["BLOCK_DELTA"] = df.groupby("RIG_ID")["BLOCK_HEIGHT"].diff().fillna(0)   # block moving = tripping / connection
    return df

S_FEATS = ["HOOKLOAD", "TORQUE", "SPP", "RPM", "ROP", "WOB", "BLOCK_HEIGHT", "FLOW_IN", "BLOCK_DELTA"]
s_tr, s_te = wide("'2026-10-02','2026-10-05','2026-10-08'"), wide("'2026-10-12'")
CLASSES = sorted(s_tr.STATE.unique())
s_model = XGBClassifier(n_estimators=120, max_depth=4, learning_rate=0.1).fit(
    s_tr[S_FEATS], s_tr.STATE.map({c: i for i, c in enumerate(CLASSES)}))
s_te["PREDICTED_STATE"] = [CLASSES[i] for i in s_model.predict(s_te[S_FEATS])]
s_acc = accuracy_score(s_te.STATE, s_te.PREDICTED_STATE)
with exp.start_run(f"rig_state_xgb_{time.strftime('%Y%m%d_%H%M%S')}"):
    exp.log_metrics({"accuracy": s_acc})
print(f"held-out accuracy {s_acc:.3f}")
pd.DataFrame(confusion_matrix(s_te.STATE, s_te.PREDICTED_STATE, labels=CLASSES), index=CLASSES, columns=CLASSES)

In [ ]:
reg.log_model(s_model, model_name="RIG_STATE_CLASSIFIER", version_name="V1",
              sample_input_data=s_tr[S_FEATS].head(50), target_platforms=["WAREHOUSE"],
              metrics={"accuracy": s_acc, "classes": CLASSES},
              comment="XGBoost rig state per minute; class index order: " + ", ".join(CLASSES))
session.write_pandas(s_te[["RIG_ID", "TS", "STATE", "PREDICTED_STATE"]].rename(columns={"STATE": "GIVEN_STATE"}),
                     "RIG_STATE_PREDICTIONS", database="HOL_PTEN_ML", schema="PREDICTIONS",
                     auto_create_table=True, overwrite=True, use_logical_type=True)

### Serve the stuck-pipe model from SPCS for live scoring

In a warehouse, each call to the model starts a Python runtime, about **12 seconds** before the first
score. That is fine for batch, too slow for an agent answering a person. Deploying the same
registered version as an **inference service** on SPCS keeps it warm: about **3 seconds** per model call
from SQL (a full agent re-score answer is 18 to 22 s warm). The agent's `ScoreRigNow` tool calls `STUCK_PIPE_SVC!PREDICT_PROBA`.

> **Why it matters:** Same model, same registry version, two serving modes. Batch scoring stays in the warehouse; anything a person is waiting on goes to a warm service. Nobody rebuilds or re-exports the model.
>
> **Two ways to run this step**
> - **Option A, plain Python/SQL:** run the cell below (the image build takes about 6 minutes and the cell waits for it; talk through the agent slides meanwhile).
> - **Option B, Cortex Code (CoCo):** open CoCo (Snowsight Workspaces, Desktop or CLI), the `machine-learning` skill
>   loads automatically, then paste:
>
> ```text
> Deploy HOL_PTEN_ML.ML.STUCK_PIPE_MODEL V1 as an SPCS inference service HOL_PTEN_ML.ML.STUCK_PIPE_SVC on compute pool HOL_PTEN_POOL using image repository HOL_PTEN_ML.APP.IMAGES, no public ingress, one instance.
> ```
>
> Both paths create the same object names, so you can mix them and the later cells still work.

In [ ]:
mv = reg.get_model("STUCK_PIPE_MODEL").version("V1")
if not q("SHOW SERVICES LIKE 'STUCK_PIPE_SVC' IN SCHEMA HOL_PTEN_ML.ML").shape[0]:
    mv.create_service(service_name="HOL_PTEN_ML.ML.STUCK_PIPE_SVC", service_compute_pool="HOL_PTEN_POOL",
                      image_repo="HOL_PTEN_ML.APP.IMAGES", ingress_enabled=False, max_instances=1, block=False)
# block=False returns while the image build job runs; the service object only appears after
# the build (about 6 min). Wait for it, then keep it up for the session: the default idle
# suspend is 30 min and a cold restart adds ~80 s to the first ScoreRigNow call.
import time
for _ in range(90):
    if q("SHOW SERVICES LIKE 'STUCK_PIPE_SVC' IN SCHEMA HOL_PTEN_ML.ML").shape[0]:
        break
    time.sleep(10)
q("ALTER SERVICE HOL_PTEN_ML.ML.STUCK_PIPE_SVC SET AUTO_SUSPEND_SECS = 14400")
q("SHOW SERVICES IN SCHEMA HOL_PTEN_ML.ML")

> **Exercise for later: wire the rig-state model into the features**
>
> Replace the given `state` column with `RIG_STATE_CLASSIFIER` predictions when building `STAND_STATE_TAG_AGG`, retrain the stuck-pipe model, and compare PR-AUC. That is the honest production setup.

## A4. The DSA agent  *(Run now, about 15 min)*

A Cortex Agent plans which tool to use for each part of a question. We give it three tools:

| Tool | Type | Answers |
|---|---|---|
| `DrillingAnalyst` | Cortex Analyst over a **semantic view** | "Which rigs, which basin, how many high-risk stands, how much NPT? Slowest connections? Best MSE? Is drag rising?" |
| `GetTagWindow` | Custom tool: stored procedure | "Show torque on RIG_007 while drilling between 06:00 and 12:00": the **selective tag retrieval** pattern (rig + tags + state + window) from the Oct 23 discussion |
| `ScoreRigNow` | Custom tool: calls the model service | "Re-score the last 5 stands on RIG_003" |
| *(slot)* | Cortex Search | Documents: added in **Lab B** |

The semantic view is the business vocabulary: it tells Analyst that "risk" means
`STUCK_PIPE_RISK`, that "flat time" means connection minutes, that a rig belongs to a basin, and
how scores, MSE, connections and drag join to rigs.

> **Why it matters:** This is where everything comes together: the data, features, model and insights become something an engineer can just ask about, in drilling language, with every answer running under their own role.
>
> **Two ways to run this step**
> - **Option A, plain Python/SQL:** run the cells below (the first one runs `lab-a-ml-to-agent/dsa_agent.sql`).
> - **Option B, Cortex Code (CoCo):** open CoCo (Snowsight Workspaces, Desktop or CLI), the `agent-studio` skill
>   loads automatically, then paste:
>
> ```text
> Create semantic view HOL_PTEN_ML.AGENT.DSA_DRILLING_SV over AGG.WELL_HEADER, PREDICTIONS.STUCK_PIPE_SCORES, LABELS.NPT_EVENTS, AGG.STAND_MSE, AGG.CONNECTION_TIMES and AGG.OVERPULL_TREND joined on RIG_ID, then create Cortex Agent HOL_PTEN_ML.AGENT.DSA_AGENT with that semantic view as DrillingAnalyst, procedure AGENT.GET_TAG_WINDOW as GetTagWindow and AGENT.SCORE_RIG_NOW as ScoreRigNow, on warehouse HOL_PTEN_WH.
> ```
>
> Both paths create the same object names, so you can mix them and the later cells still work.

In [ ]:
# Semantic view, two tool procedures and the agent: runs lab-a-ml-to-agent/dsa_agent.sql statement by statement (open the file to read the SQL)
run_sql_file("lab-a-ml-to-agent/dsa_agent.sql")

In [ ]:
import json

def ask_agent(question, agent="HOL_PTEN_ML.AGENT.DSA_AGENT"):
    payload = json.dumps({"messages": [{"role": "user", "content": [{"type": "text", "text": question}]}]})
    raw = session.sql("SELECT SNOWFLAKE.CORTEX.DATA_AGENT_RUN(?, ?, FALSE)", params=[agent, payload]).collect()[0][0]
    content = json.loads(raw).get("content", [])
    tools = [c["tool_use"]["name"] for c in content if c.get("type") == "tool_use"]
    print("Tools used:", tools, "\n")
    print("\n".join(c["text"] for c in content if c.get("type") == "text"))

ask_agent("Which 5 rigs had the highest stuck-pipe risk on October 12?")

In [ ]:
ask_agent("Show torque and hookload for RIG_007 while drilling on 2026-10-10 between 06:00 and 12:00")

In [ ]:
ask_agent("Which rigs have the slowest median connection time versus the fleet, and how many hours is that?")

In [ ]:
ask_agent("Is RIG_007 showing rising drag before its stuck-pipe risk spikes?")

**Go look:** Snowsight » **AI & ML » Agents » DSA_AGENT**. Ask a question in the playground and
open the **trace**: you can see the plan, each tool call, the SQL Analyst generated, and the
latency of each step. That trace is the native observability you will contrast with LangChain +
TruLens in Lab B.

**A quick evaluation:** agents need a regression test like any other code. Keep a small set of
questions with expected answers and run them after every change to the semantic view or instructions.

In [ ]:
EVAL = [
    ("How many STUCK_PIPE events were recorded in total?",
     q("SELECT COUNT(*) n FROM LABELS.NPT_EVENTS WHERE event_type='STUCK_PIPE'").N[0]),
    ("Which basin is RIG_007 in?", q("SELECT basin FROM AGG.WELL_HEADER WHERE rig_id='RIG_007'").BASIN[0]),
]
for question, expected in EVAL:
    print(f"Q: {question}\n   expected: {expected}")
    ask_agent(question); print("-" * 80)

> **Vendor agnostic vs Snowflake native: the agent layer**
>
> | | Vendor agnostic | Snowflake native |
> |---|---|---|
> | **How** | LangGraph/CrewAI agent in Python, calling a SQL tool, a model endpoint and a vector DB | Cortex Agent: semantic views, custom tools, search, with tracing and evaluations built in |
> | **Pros** | Maximum flexibility; swap LLM providers freely | Runs under the user's role so RBAC applies to every tool; no credentials to manage; traces in Snowsight |
> | **Cons** | You own planning, tool auth, retries, tracing and evaluation; credentials for every tool; data leaves through prompts | Orchestration choices are Snowflake's; model choice limited to Cortex models |
>
> **When to pick which:** start native for anything that answers questions over Snowflake data. Reach for a framework when the workflow spans many external systems; the native agent can still be one of its tools (REST/MCP).

> **Exercise for later: evaluate at scale**
>
> Write 20 real DSA questions with expected answers, run them through Snowsight's agent evaluations (on the DSA_AGENT page in Snowsight), and add verified queries to the semantic view for the ones it gets wrong.

## A5. Serve the agent from a container (SPCS)  *(Run now, about 7 min)*

The team wants an app, not a notebook. `app/` is the **PTEN+ DSA** app: a FastAPI service with a
fleet dashboard (KPI tiles, risk by rig, risk and drag trend, drilling channels, connection
benchmark, MSE vs ROP, rig-state classifier, NPT by basin) next to an embedded agent chat. Clicking a
rig on any chart makes it the agent's context.

- In SPCS it authenticates with the service's own OAuth token, so **no secrets** live in the image,
  and the endpoint sits behind Snowflake login.
- It calls the agent through the **Cortex Agents REST API** (`.../agents/DSA_AGENT:run`), which
  returns a request id per answer.
- Every answer has **thumbs up / down**. The app posts it to the **Feedback REST API**
  (`.../agents/DSA_AGENT:feedback`) with that request id, so feedback lands next to the trace in
  **AI & ML » Agents » DSA_AGENT » Observability**, where the team can review bad answers and fix
  the semantic view or instructions.

**This is the one step that needs your laptop.** A notebook cannot build a container image, so
the image is built with Docker or Podman and the Snowflake CLI, then pushed to your account's
image repository. The repository URL is printed by the first cell below.

```bash
git clone https://github.com/sfc-gh-srahman/pten-hol && cd pten-hol
snow spcs image-registry login -c <connection>
docker build --platform linux/amd64 -t <repository_url>/dsa-app:v8 lab-a-ml-to-agent/app   # or podman
docker push <repository_url>/dsa-app:v8
```

> **No Docker or Podman? That's fine.** Skip the rest of A5 and follow along on the presenter's
> screen. Nothing later depends on the app: the agent works the same in Snowsight, and Lab B runs as
> written.

> **Why it matters:** A notebook proves it works; an app is what a superintendent will open. Same agent, same governance, and every thumbs-down becomes a work item for the DSA team instead of a complaint in a hallway.
>
> **Two ways to run this step**
> - **Option A, plain Python/SQL:** build the image as shown, then run the cells below (they run `lab-a-ml-to-agent/deploy_app.sql`).
> - **Option B, Cortex Code (CoCo):** open CoCo (Snowsight Workspaces, Desktop or CLI), the `snowflake-apps` skill
>   loads automatically, then paste:
>
> ```text
> Deploy the FastAPI app in lab-a-ml-to-agent/app to SPCS as service HOL_PTEN_ML.APP.DSA_APP on compute pool HOL_PTEN_POOL with image repository HOL_PTEN_ML.APP.IMAGES, a public endpoint ui on port 8080, readiness probe /healthz, env AGENT_FQN=HOL_PTEN_ML.AGENT.DSA_AGENT, and a service role app_user granted to HOL_PTEN_ROLE.
> ```
>
> Both paths create the same object names, so you can mix them and the later cells still work.

In [ ]:
q("SHOW IMAGE REPOSITORIES IN SCHEMA HOL_PTEN_ML.APP")

In [ ]:
# Create the service and grant access: runs lab-a-ml-to-agent/deploy_app.sql statement by statement (open the file to read the SQL)
run_sql_file("lab-a-ml-to-agent/deploy_app.sql")

In [ ]:
import time
for _ in range(30):
    status = session.sql("SELECT SYSTEM$GET_SERVICE_STATUS('HOL_PTEN_ML.APP.DSA_APP')").collect()[0][0]
    if '"READY"' in status: break
    time.sleep(10)
print(status)
q("SHOW ENDPOINTS IN SERVICE HOL_PTEN_ML.APP.DSA_APP")

Open the `ingress_url` in a browser, sign in with Snowflake, ask a sample question and give the answer a
thumbs up or down. **Where it lands:**

- Snowsight » **AI & ML » Agents » DSA_AGENT » Observability**: each thread with its trace and the user feedback next to it
- The same events in SQL, via `SNOWFLAKE.LOCAL.GET_AI_OBSERVABILITY_EVENTS` (record name `CORTEX_AGENT_FEEDBACK`).
  The thumbs value is in `VALUE`; who gave it (the app's service user, or your user locally) is in `RECORD_ATTRIBUTES`.
  Viewing needs `MONITOR` on the agent plus `SNOWFLAKE.CORTEX_USER`; the feedback text needs
  `READ UNREDACTED AI OBSERVABILITY EVENTS TABLE`.
- App logs and status: **Monitoring » Services & jobs » DSA_APP**

In [ ]:
q("""
SELECT timestamp,
       record_attributes:"snow.ai.observability.user.name"::string AS user_name,
       IFF(value:positive::boolean, 'thumbs up', 'thumbs down')     AS rating,
       value:feedback_message::string                              AS message,
       ARRAY_TO_STRING(value:categories, ', ')                     AS categories,
       record_attributes:"ai.observability.record_id"::string      AS record_id   -- joins to the request trace
FROM TABLE(SNOWFLAKE.LOCAL.GET_AI_OBSERVABILITY_EVENTS('HOL_PTEN_ML', 'AGENT', 'DSA_AGENT', 'CORTEX AGENT'))
WHERE record:name = 'CORTEX_AGENT_FEEDBACK'
ORDER BY timestamp DESC LIMIT 20
""")

> **Vendor agnostic vs Snowflake native: hosting the app**
>
> | | Vendor agnostic | Snowflake native |
> |---|---|---|
> | **How** | Container on ECS/EKS or a VM; API gateway; secrets manager for Snowflake credentials | Snowpark Container Services: the same container, running next to the data |
> | **Pros** | Familiar to platform teams; any networking setup | OAuth token injected, Snowflake login on the endpoint, service roles for access, no data egress |
> | **Cons** | Credential rotation, ingress auth and egress to Snowflake are yours to build and audit | Compute pools to size; public internet exposure is through Snowflake ingress |
>
> **When to pick which:** the container is portable either way. Run it in SPCS when its main job is to talk to Snowflake data and agents.

> **Exercise for later: ship it properly**
>
> Stream agent responses to the UI (`stream: true` on `:run`), turn thumbs-down answers into new verified queries on the semantic view, and deploy from CI using `snow spcs service upgrade`.

## A6. Wrap up

You built, on PTEN-shaped data and without moving it:

1. Stand-level features in a governed Feature Store
2. A tracked, registered stuck-pipe model, batch scored in the warehouse and served live from SPCS
3. Drilling insights from the same data: MSE, connection flat time, drag, and a rig-state classifier
4. A DSA agent that combines semantic SQL, selective tag retrieval and live scoring
5. The PTEN+ DSA app: dashboards, KPI tiles and an embedded agent with end-user feedback

### Follow-ups worth driving next

These are deliberately out of scope today, because they need new data, higher-frequency signals or
inference at the rig edge. Each one reuses the same pattern: features, registry, agent tool, app panel.

| Pillar | Follow-up | What it needs |
|---|---|---|
| Drilling optimization | **Stick-slip, bit bounce, whirl** detection | 1 to 50 Hz surface torque and downhole vibration; spectral features (container runtime) |
| Drilling optimization | **Bit wear and economic trip point** | Bit runs and dull grades joined to MSE per footage |
| Wellbore stability | **Pore pressure and fracture gradient** updates | Gamma ray, resistivity and mud weight logs; offset-well models |
| Well control (HSE) | **Kick and loss detection** | Pit volume, flow out vs flow in at 1 Hz or faster; must run at the edge in under 500 ms, cloud trains and monitors |
| Well control | **Gas influx profiling** | Mud-gas chromatography by depth |
| Equipment health | **Top drive and mud pump diagnostics, drilling line RUL** | Vibration, temperature, pressure ripple; ton-mile and load-cycle history |
| Equipment health | **Mud motor / RSS failure** | Differential pressure across the motor, downhole tool data |
| Steering | **Stratigraphic boundaries, trajectory projection** | M/LWD logs, surveys, toolface and dogleg data |
| Architecture | **Irregular, delayed downhole data** | As-of joins and late-arriving data handling for mud-pulse telemetry |

**Bridge to Lab B:** the agent has an empty slot for documents. Lab B builds the retrieval
foundation (parse, chunk, Cortex Search), shows the LangChain way first, then plugs the same search
service into this agent.